# Bronze — Orders
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | ADLS `orders/` folder |
| **Target** | `{catalog}.bronze.orders` |
| **Pattern** | COPY INTO — tracks ingested files, appends only new ones |

This notebook is designed to run as a **Databricks Workflow task**.
All configuration is passed via job parameters — change only `catalog` and `source_path` in the widget defaults below.

## Setup — Widgets & Constants

Widget values are overridden at runtime by Workflow job parameters.
Running the notebook manually uses the defaults defined here.

In [ ]:
dbutils.widgets.text('catalog',     'your_catalog')
dbutils.widgets.text('schema',      'bronze')
dbutils.widgets.text('source_path', 'abfss://raw-data@<storage-account>.dfs.core.windows.net/orders/')
dbutils.widgets.text('batch_id',    'batch_001')

CATALOG     = dbutils.widgets.get('catalog')
SCHEMA      = dbutils.widgets.get('schema')
SOURCE_PATH = dbutils.widgets.get('source_path')
BATCH_ID    = dbutils.widgets.get('batch_id')
TABLE       = f'{CATALOG}.{SCHEMA}.orders'

print(f'Target table : {TABLE}')
print(f'Source path  : {SOURCE_PATH}')
print(f'Batch ID     : {BATCH_ID}')

## Step 1 — Create Bronze Table

Creates the table on first run. Skipped on subsequent runs — `IF NOT EXISTS` makes this safe to re-run.
All source columns land as STRING. Casting happens in Silver.

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE} (
        order_id     STRING,
        customer_id  STRING,
        order_date   STRING,
        status       STRING,
        created_at   STRING,
        _ingested_at TIMESTAMP,
        _source_file STRING,
        _batch_id    STRING
    )
    USING DELTA
''')

print(f'Table ready: {TABLE}')

## Step 2 — Ingest with COPY INTO

COPY INTO scans the source folder and loads only files not yet ingested.
The subquery form lets us add audit columns at ingest time using `_metadata.file_path`.

The result shows `num_files_copied` and `num_rows_inserted` — if both are 0, no new files were found.

In [ ]:
result = spark.sql(f'''
    COPY INTO {TABLE}
    FROM (
        SELECT
            order_id,
            customer_id,
            order_date,
            status,
            created_at,
            current_timestamp()  AS _ingested_at,
            _metadata.file_path  AS _source_file,
            \'{BATCH_ID}\'        AS _batch_id
        FROM \'{SOURCE_PATH}\'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        \'header\'      = \'true\',
        \'inferSchema\' = \'false\'
    )
''')

result.display()

## Step 3 — Verify

In [ ]:
bronze_df = spark.table(TABLE)
print(f'Total rows in {TABLE}: {bronze_df.count():,}')
bronze_df.orderBy('_ingested_at', ascending=False).display()